# BC_515_S1 | Figure 5

Saved SHINE results. Training seed 123; K=7; clustering seed 0.

Five-fit summaries use seeds 0, 1, 2, 123 and 1234. See [setup and data requirements](README.md).


In [72]:
import os
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import shine
from shine import SHINE, normalize_counts
from shine.study import Section

CASE = "BC_515_S1"
# Extract the figure-data archive into data/, or set SHINE_DATA_ROOT.
search_roots = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("SHINE_DATA_ROOT"):
    ROOT = Path(os.environ["SHINE_DATA_ROOT"]).expanduser().resolve()
else:
    ROOT = next((candidate for parent in search_roots
                 for candidate in (parent / "data", parent)
                 if (candidate / "revision_experiments").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Extract the figure-data archive into data/, or set SHINE_DATA_ROOT. See data/README.md.")
manifest_candidates = [p / name for p in search_roots
                       for name in ("sections.json", "notebooks/sections.json")]
manifest_candidates.append(ROOT / "SHINE-github/notebooks/sections.json")
MANIFEST = next((p for p in manifest_candidates if p.is_file()), None)
if MANIFEST is None:
    raise FileNotFoundError("Cannot find notebooks/sections.json.")
import sys
sys.path.insert(0, str(MANIFEST.parent))
from figure_plotting import spatial_results, regional_results, molecular_maps, attention_plot, save_panel

OUT = ROOT / "outputs/notebook_results_release_011_20261127" / CASE
OUT.mkdir(parents=True, exist_ok=True)
section = Section(ROOT, MANIFEST, CASE)
print("shine-omics", shine.__version__, "|", CASE, "|", len(section.spots), "paired spots")

shine-omics 0.1.1 | BC_515_S1 | 3508 paired spots


## 1. Settings

Fixed figure settings and paired cohort. Source files are verified by SHA-256.

In [73]:
settings = {"Figure": section.config["figure"], "K": section.config["K"],
            "epochs": 120, "training_seed": 123,
            "clustering_seed": section.config["display_clustering_seed"],
            "local_mask": True, "local_neighbors": section.config["graph_options"]["local_k"],
            "spatial_graph_neighbors": section.config["graph_options"]["spatial_k"], "final_ReLU": True, **section.config["model_options"]}
display(pd.Series(settings, name="Final setting").to_frame())
display(section.spots.groupby(["region", "lesion"], dropna=False).size().rename("paired_spots").reset_index())

                          Final setting
Figure                                5
K                                     7
epochs                              120
training_seed                       123
clustering_seed                       0
local_mask                         True
local_neighbors                       6
spatial_graph_neighbors               6
final_ReLU                         True
loss                     reconstruction
l2                                False
spatial_floor                       0.0
latent_dim                          128
attention_backend                 dense
evaluate_zero_alignment           False

                  region  lesion  paired_spots
0                   CAFs     NaN           705
1   Cancer Basal/Her2 SC     NaN           166
2         Cancer Cycling     NaN             3
3       Cancer LumA/B SC     NaN           237
4                 Immune     NaN           730
5      Normal epithelial     NaN            81
6  Other (stroma region)     NaN           260
7   Other (tumor region)     NaN           318
8             Unlabelled     NaN            77
9               Vascular     NaN           931

## 2. Inputs

These paired matrices are already preprocessed. Do not repeat normalization or registration.

In [74]:
print(section.config["preprocessing"])
input_audit = pd.DataFrame([
    {"modality": name, "spots": x.shape[0], "features": x.shape[1],
     "finite": bool(np.isfinite(x).all()), "constant_rows": int((x.std(1) == 0).sum()),
     "minimum": float(x.min()), "maximum": float(x.max())}
    for name, x in [("RNA", section.rna), ("MSI", section.msi)]])
display(input_audit)
assert section.spots.spot.is_unique
assert np.allclose(section.xy, section.spots[["x", "y"]])
# For raw counts: normalize_counts(full_counts, feature_indices=frozen_columns)
# Do not renormalize section.rna or section.msi.

Frozen original notebook paired matrices; RNA QC and HVG selection, RNA/MSI total/TIC 10000/log1p were done upstream. No repeat normalization. Features and barcodes are hash-verified.


  modality  spots  features  finite  constant_rows  minimum   maximum
0      RNA   3508      2000    True              0      0.0  8.367816
1      MSI   3508      2082    True              0      0.0  7.367380

## 3. Graphs

H5 + H20; spatial kNN6; feature correlation-kNN20; local self+6 attention. Graphs use no annotation labels.

In [75]:
data, graphs = section.prepare()
display(pd.Series(graphs["audit"], name="Graph audit").to_frame())
print("Validated tensor shape:", data.shape)
print("Input + graph fingerprint:", data.fingerprint())

Validated tensor shape: (3508, 2000, 2082)
Input + graph fingerprint: 4f73a02898fa15c3fde4910cc81ee13b2cf82d7399b8f41768eea7f4c3d45450


                                     Graph audit
n_spots                                     3508
spatial_k                                      6
feature_k                                     20
local_mask                                  True
local_k                                        6
cross_modal_support          self_plus_neighbors
hyperedge_sizes                          [5, 20]
n_hyperedges                                7016
pca_components                                30
pca_seed                                       0
constant_feature_rows                     [0, 0]
allow_constant_feature_rows                 True
labels_used                                False
sparse_propagation                         False

## 4. Model

Replay the saved figure model. Set `RUN_NEW_TRAINING=True` for a separate 120-epoch fit.

In [76]:
model, labels, replay_audit = section.replay(data)
display(pd.Series(replay_audit).to_frame("Replay check"))
RUN_NEW_TRAINING = False
if RUN_NEW_TRAINING:
    new_model = SHINE(**section.config["model_options"])
    def progress(record):
        if record["epoch"] % 20 == 0:
            print("Epoch", record["epoch"], "loss", round(record["total"], 5))
    new_model.fit(data, epochs=120, seed=123, lr=1e-4, weight_decay=1e-3, progress=progress)
    new_labels = new_model.cluster(section.config["K"], seed=section.config["display_clustering_seed"])
    new_model.save(OUT / "new_training_seed123.pt")
    display(section.score(new_labels))
else:
    print("Displaying the saved figure model.")

Displaying the saved figure model.


                                                           Replay check
embedding_max_abs_error                                             0.0
figure_labels_exact                                                True
n_spots                                                            3508
origin                   saved figure checkpoint; new package inference

## 5. ARI and NMI

Average five clustering seeds within each fit, then report mean and sample SD over five training seeds. Regional scores retain whole-section cluster IDs.

In [77]:
display_metrics = section.score(labels)
raw_metrics, per_fit, summary = section.five_fit_metrics()
print("Displayed fit: seed 123")
display(display_metrics.round(4))
print("Five fits: mean and sample SD")
display(summary.round(4))
display(per_fit.round(4))
for name, table in [("display_metrics", display_metrics), ("all_partitions", raw_metrics),
                    ("per_training_fit", per_fit), ("five_fit_summary", summary)]:
    table.to_csv(OUT / (name + ".csv"), index=False)

Displayed fit: seed 123
Five fits: mean and sample SD


        endpoint  n_scored     ARI     NMI
0  Whole section      3508  0.2798  0.3586

        endpoint  ARI_mean  ARI_std  NMI_mean  NMI_std  n_fits
0  Whole section    0.3075   0.0189    0.3652   0.0076       5

        endpoint  training_seed     ARI     NMI
0  Whole section              0  0.3159  0.3675
1  Whole section              1  0.3078  0.3637
2  Whole section              2  0.3305  0.3769
3  Whole section            123  0.2788  0.3578
4  Whole section           1234  0.3047  0.3599

## 6. Spatial partition and UMAP

Manuscript colors and spatial rendering; seed 123. Filled display cells remove scatter gaps without changing measured spots or cluster IDs. Grey denotes unlabelled or unpaired tissue. UMAP is used only for visualization.


In [78]:
import umap
saved_embedding = np.load(section.path("embedding_123"))
umap_xy = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=123, n_jobs=1).fit_transform(saved_embedding)
fig = spatial_results(section, labels, umap_xy)
save_panel(fig, OUT / "seed123_spatial_UMAP")
plt.show()
spatial_source = section.spots.copy()
spatial_source["SHINE_cluster"] = labels
spatial_source[["UMAP1", "UMAP2"]] = umap_xy
spatial_source.to_csv(OUT / "seed123_spatial_source.csv", index=False)

<Figure size 1050x410 with 3 Axes>

## 7. Attention

Spatial-branch weights within RNA and MSI. These are mixing weights, not causal importance.

In [79]:
fig = attention_plot(model)
save_panel(fig, OUT / "attention_weights")
plt.show()
attention_source = pd.DataFrame({"spot": section.spots.spot,
    "RNA_spatial_raw": model.attention_["rna_raw"][:, 0],
    "RNA_spatial_effective": model.attention_["rna"][:, 0],
    "MSI_spatial_raw": model.attention_["msi_raw"][:, 0],
    "MSI_spatial_effective": model.attention_["msi"][:, 0]})
attention_source.to_csv(OUT / "attention_source.csv", index=False)
display(attention_source.drop(columns="spot").describe().round(3))

<Figure size 720x260 with 2 Axes>

       RNA_spatial_raw  ...  MSI_spatial_effective
count         3508.000  ...               3508.000
mean             0.005  ...                  0.003
std              0.003  ...                  0.000
min              0.002  ...                  0.002
25%              0.003  ...                  0.003
50%              0.004  ...                  0.003
75%              0.006  ...                  0.003
max              0.019  ...                  0.006

[8 rows x 4 columns]

## 8. Measured genes and MSI peaks

Six genes and six peaks with the largest positive rank-biserial effects for tumor-enriched cluster C1 versus other spots. Peak identities require separate evidence.

In [80]:
molecules = section.molecular_values()
print("RNA:", section.config["display_genes"])
print("MSI m/z:", section.config["display_peaks"])
fig = molecular_maps(section, molecules)
save_panel(fig, OUT / "measured_molecules")
plt.show()
measured_source = pd.concat([section.spots[["spot", "x", "y", "region", "lesion"]], molecules], axis=1)
measured_source.to_csv(OUT / "measured_molecule_source.csv", index=False)
effects = section.molecular_comparison(molecules, labels)
display(effects.round(3))
effects.to_csv(OUT / "measured_molecule_effects.csv", index=False)

RNA: ['AGR2', 'ALOX15B', 'GOLM1', 'MUC19', 'CRABP2', 'HMGCS2']
MSI m/z: ['mz_794.5445377570832', 'mz_795.5449847802124', 'mz_796.5468817265635', 'mz_744.552634988283', 'mz_848.603475163387', 'mz_738.6601470143742']


<Figure size 1200x450 with 13 Axes>

          contrast               feature  ...  mean_target  difference_SD
0   C1 versus rest                  AGR2  ...        3.684          1.598
1   C1 versus rest               ALOX15B  ...        2.227          1.992
2   C1 versus rest                 GOLM1  ...        4.420          1.475
3   C1 versus rest                 MUC19  ...        3.540          1.233
4   C1 versus rest                CRABP2  ...        1.948          1.638
5   C1 versus rest                HMGCS2  ...        2.194          1.741
6   C1 versus rest  mz_794.5445377570832  ...        5.424          1.873
7   C1 versus rest  mz_795.5449847802124  ...        4.662          1.607
8   C1 versus rest  mz_796.5468817265635  ...        4.424          1.704
9   C1 versus rest   mz_744.552634988283  ...        4.427          1.751
10  C1 versus rest   mz_848.603475163387  ...        3.519          1.751
11  C1 versus rest  mz_738.6601470143742  ...        3.447          1.979

[12 rows x 7 columns]

## 9. Molecular selection

Selected cluster, effect direction and ranking from the figure source tables.

In [81]:
for modality in ["rna", "msi"]:
    section.verify(modality + "_selection")
    print(modality.upper())
    display(pd.read_csv(section.path(modality + "_selection")))

RNA
MSI


   cluster  feature  rank_effect  mean_difference  standardized_difference
0        1     AGR2     0.909083         2.249582                 1.597622
1        1  ALOX15B     0.832576         1.895025                 1.991734
2        1    GOLM1     0.812506         2.059426                 1.475477
3        1    MUC19     0.778916         1.703214                 1.232709
4        1   CRABP2     0.745828         1.557661                 1.637605
5        1   HMGCS2     0.722986         1.786566                 1.740970

   cluster               feature  ...  mean_difference  standardized_difference
0        1  mz_794.5445377570832  ...         1.207766                 1.872623
1        1  mz_795.5449847802124  ...         1.143138                 1.606831
2        1  mz_796.5468817265635  ...         1.101191                 1.704502
3        1   mz_744.552634988283  ...         1.161238                 1.750690
4        1   mz_848.603475163387  ...         1.940940                 1.751132
5        1  mz_738.6601470143742  ...         1.597962                 1.979064

[6 rows x 5 columns]

## 10. Validation and exports

Saved-model replay and fresh training are reported separately. Fresh fits do not replace the figure model.

In [82]:
validation_dir = ROOT / "outputs/shine_omics_notebooks_validation_20261124" / CASE
for filename in ["replay_validation.json", "fresh_training_validation.json"]:
    path = validation_dir / filename
    if path.exists():
        display(pd.Series(json.loads(path.read_text()), name=filename).to_frame())
    else:
        print(filename, "Not available; training validation is incomplete.")
provenance = {"case": CASE, "package_version": shine.__version__, "settings": settings,
              "replay": replay_audit, "sources": section.config["sources"],
              "statistical_unit": "training seed; sample SD over five fits",
              "spatial_train_seed": 123, "new_training_replaces_figure": False}
(OUT / "notebook_provenance.json").write_text(json.dumps(provenance, indent=2))
print("Output directory:", OUT)

Output directory: /Users/dubingxue/Desktop/SHINE/outputs/notebook_results_release_011_20261127/BC_515_S1


                                                    replay_validation.json
embedding_max_abs_error                                                0.0
figure_labels_exact                                                   True
n_spots                                                               3508
origin                      saved figure checkpoint; new package inference
graph_audit              {'n_spots': 3508, 'spatial_k': 6, 'feature_k':...
data_fingerprint         4f73a02898fa15c3fde4910cc81ee13b2cf82d7399b8f4...
seconds                                                          48.610345

                                  fresh_training_validation.json
case                                                   BC_515_S1
epochs                                                       120
training_seed                                                123
seconds                                              1029.965002
finite_embedding                                            True
fresh_vs_historical_max_abs_error                       0.000003
partition_ARI_vs_historical                                  1.0
main_figure_overwritten                                    False